# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hussainhhgh/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit


hf_token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')"
)

month_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)


feature_query = f"""
    SELECT
        content_hash_id,
        client_hash_id,

        SUM(gsc_impressions) AS impressions_15d,
        SUM(gsc_clicks) AS clicks_15d,
        AVG(gsc_avg_position) AS avg_position_15d

    FROM read_parquet('{month_path}')

    WHERE report_date <= DATE '2026-03-15'
      AND gsc_data_available = TRUE

    GROUP BY
        content_hash_id,
        client_hash_id

    HAVING SUM(gsc_impressions) > 0
"""

features_df = con.sql(feature_query).df()


label_query = f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_clicks) AS clicks_1631

    FROM read_parquet('{month_path}')

    WHERE report_date >= DATE '2026-03-16'
      AND gsc_data_available = TRUE

    GROUP BY
        content_hash_id,
        client_hash_id
"""

labels_df = con.sql(label_query).df()


merged = features_df.merge(
    labels_df,
    on=["content_hash_id", "client_hash_id"],
    how="left"
)


merged["ctr_15d"] = (
    merged["clicks_15d"] /
    merged["impressions_15d"]
)

merged["log_impressions_15d"] = np.log1p(
    merged["impressions_15d"]
)

merged["log_clicks_15d"] = np.log1p(
    merged["clicks_15d"]
)


merged["declining"] = np.where(
    merged["clicks_1631"].notna(),
    (
        merged["clicks_1631"] <
        merged["clicks_15d"] * 0.90
    ).astype(int),
    np.nan
)


def position_bucket(pos):
    if pos <= 3:
        return "top_3"
    elif pos <= 10:
        return "page_1"
    elif pos <= 20:
        return "striking"
    elif pos <= 50:
        return "page_3_5"
    else:
        return "deep"

merged["position_tier"] = (
    merged["avg_position_15d"]
    .apply(position_bucket)
)


features = [
    "log_impressions_15d",
    "log_clicks_15d",
    "avg_position_15d",
    "ctr_15d"
]

# Only rows with a known future outcome are eligible for modeling.
model_df = merged.dropna(
    subset=features + ["declining"]
).copy()

model_df["declining"] = (
    model_df["declining"]
    .astype(int)
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        model_df,
        groups=model_df["client_hash_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("=== DATA REBUILD ===")
print(f"Feature rows: {len(features_df):,}")
print(f"Future-label rows: {len(labels_df):,}")
print(f"Merged rows: {len(merged):,}")
print(
    f"Rows with observed future outcome: "
    f"{merged['declining'].notna().sum():,}"
)
print(
    f"Rows with unknown future outcome: "
    f"{merged['declining'].isna().sum():,}"
)

print("\n=== MODEL DATA ===")
print(f"Rows used for modeling: {len(model_df):,}")
print(f"Decline rate: {model_df['declining'].mean():.3f}")

print("\n=== SPLIT ===")
print(
    f"Train: {len(train_df):,} rows, "
    f"{train_df['client_hash_id'].nunique()} clients"
)
print(
    f"Test: {len(test_df):,} rows, "
    f"{test_df['client_hash_id'].nunique()} clients"
)

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

print(
    f"Client overlap: "
    f"{len(train_clients & test_clients)}"
)

print(
    f"position_tier in train_df columns: "
    f"{'position_tier' in train_df.columns}"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== DATA REBUILD ===
Feature rows: 151,981
Future-label rows: 166,224
Merged rows: 151,981
Rows with observed future outcome: 141,467
Rows with unknown future outcome: 10,514

=== MODEL DATA ===
Rows used for modeling: 141,467
Decline rate: 0.198

=== SPLIT ===
Train: 129,698 rows, 30 clients
Test: 11,769 rows, 13 clients
Client overlap: 0
position_tier in train_df columns: True


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method choice and why:** Logistic Regression and Random Forest, compared side by side. My lane is scoring pages by decline risk, and both models output a probability I can rank by.

**Features:** `log_impressions_15d`, `log_clicks_15d`, `avg_position_15d`, `ctr_15d`

Impressions was the strongest, cleanest signal identified in the signal audit — decline risk rose from 2.6% to 41.4% across volume tiers — so it anchors this feature set.

**Why these two models:**
- **Logistic Regression** gives interpretable coefficients as a linear baseline
- **Random Forest** tests whether capturing feature interactions (e.g. high volume combined with poor position) meaningfully beats it, without requiring the hand-picked bucket thresholds the Week-4 baseline rule depended on

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

features = ['log_impressions_15d', 'log_clicks_15d', 'avg_position_15d', 'ctr_15d']

print(f"Features selected: {features}")
print(f"Rationale: log_impressions_15d led the signal audit with decline risk rising")
print(f"from 2.6% to 41.4% across volume tiers — the cleanest signal found.")

logreg = LogisticRegression(max_iter=1000)
rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42)

print(f"\nModels initialized: {logreg.__class__.__name__}, {rf.__class__.__name__}")

Features selected: ['log_impressions_15d', 'log_clicks_15d', 'avg_position_15d', 'ctr_15d']
Rationale: log_impressions_15d led the signal audit with decline risk rising
from 2.6% to 41.4% across volume tiers — the cleanest signal found.

Models initialized: LogisticRegression, RandomForestClassifier


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split design:** The data is split by `client_hash_id`, not by individual rows. The corrected split contains 30 clients in training and 13 previously unseen clients in testing, with zero client overlap. This grouped design is more appropriate than a row-level split because multiple pages from the same client can share client-specific characteristics; keeping those pages within the same split reduces the risk that the model learns client-specific patterns that would not generalize to unseen clients.

The resulting split contains 129,698 training rows and 11,769 test rows. Because the split is performed at the client level, the number of rows is not expected to be evenly distributed between training and test sets. The test clients also vary substantially in size, so the evaluation should be interpreted as performance across unseen clients rather than as a simple random 30% row sample.

This provides a client-held-out evaluation of whether the ranking model can generalize beyond the clients used during training.

In [4]:
from sklearn.model_selection import GroupShuffleSplit

features = ['log_impressions_15d', 'log_clicks_15d', 'avg_position_15d', 'ctr_15d']
model_df = merged.dropna(subset=features + ['declining']).copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]

print(f"Train: {len(train_df)} rows, {train_df['client_hash_id'].nunique()} clients")
print(f"Test: {len(test_df)} rows, {test_df['client_hash_id'].nunique()} clients")

Train: 129698 rows, 30 clients
Test: 11769 rows, 13 clients


In [8]:
print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", train_df["client_hash_id"].nunique())
print("Test clients:", test_df["client_hash_id"].nunique())

overlap = (
    set(train_df["client_hash_id"])
    & set(test_df["client_hash_id"])
)

print("Client overlap:", len(overlap))

print("\nRows by client:")
print(
    test_df["client_hash_id"]
    .value_counts()
    .describe()
)

Train rows: 129698
Test rows: 11769
Train clients: 30
Test clients: 13
Client overlap: 0

Rows by client:
count      13.000000
mean      905.307692
std      1269.815760
min        10.000000
25%        36.000000
50%        99.000000
75%      1293.000000
max      3343.000000
Name: count, dtype: float64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as the Week-4 baseline. The comparison is evaluated on the same client-grouped held-out test set.*

**Results:** All methods were evaluated on the same client-grouped test split, with 30 clients in training and 14 unseen clients in testing. The corrected full-warehouse pipeline produced the following results:

| Method | Precision | Recall | F1 | ROC AUC | P@10 | P@50 | P@200 |
|---|---:|---:|---:|---:|---:|---:|---:|
| Baseline rule (train-calibrated, test-scored) | 0.074 | 0.169 | 0.103 | 0.294 | 0.30 | 0.48 | 0.445 |
| Logistic Regression | 0.601 | 0.869 | 0.710 | 0.900 | 1.00 | 0.84 | 0.720 |
| Decision Tree (shallow) | 0.588 | 1.000 | 0.741 | 0.905 | 0.40 | 0.62 | 0.785 |
| Random Forest | 0.588 | 1.000 | 0.740 | **0.908** | 0.90 | **0.90** | **0.820** |

The baseline rule performs poorly on the held-out clients, with ROC AUC = 0.294 and P@50 = 0.48. All three supervised models substantially outperform the baseline. Random Forest provides the strongest overall ranking performance, achieving the highest ROC AUC (0.908), highest P@50 (0.90), and highest P@200 (0.820). Logistic Regression achieves the strongest P@10 (1.00), but Random Forest performs better at the larger review-queue sizes that are more representative of an operational content-prioritization workflow.

**Caveat worth noting:** Decision Tree and Random Forest both achieve recall = 1.000 at the default 0.5 classification threshold. This means the default binary threshold is not especially informative for judging these models as calibrated classifiers and can make F1 appear stronger than the underlying decision quality. Because this capstone is a ranking task, ROC AUC and Precision@K are treated as the primary evaluation measures, while threshold-based precision, recall, and F1 are reported as secondary diagnostics.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

# --- 1. Baseline rule, calibrated on train only, scored on test ---
band_avg_ctr_train = train_df.groupby('position_tier')['ctr_15d'].mean()
overall_train_avg = train_df['ctr_15d'].mean()

t = test_df.copy()
t['expected_ctr'] = t['position_tier'].map(band_avg_ctr_train).fillna(overall_train_avg)
decent_position = t['avg_position_15d'] <= 20
ctr_underperforming = t['ctr_15d'] < (t['expected_ctr'] * 0.7)
t['baseline_score'] = decent_position.astype(int) * ctr_underperforming.astype(int) * t['impressions_15d']
t['baseline_action'] = (t['baseline_score'] > 0).astype(int)

print('Baseline action counts on test set:')
print(t['baseline_action'].value_counts())

# --- 2. Features + models ---
num_features = ['log_impressions_15d', 'log_clicks_15d', 'avg_position_15d', 'ctr_15d']
cat_features = ['position_tier']

X_train = train_df[num_features + cat_features]
y_train = train_df['declining']
X_test = test_df[num_features + cat_features]
y_test = test_df['declining']

pre = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
])

models = {
    'logistic_regression': Pipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))]),
    'decision_tree_shallow': Pipeline([('pre', pre), ('clf', DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42))]),
    'random_forest': Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1))]),
}

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

rows = []
y_test_arr = y_test.values

rows.append({
    'method': 'Baseline rule (train-calibrated, test-scored)',
    'precision': precision_score(y_test_arr, t['baseline_action'], zero_division=0),
    'recall': recall_score(y_test_arr, t['baseline_action'], zero_division=0),
    'f1': f1_score(y_test_arr, t['baseline_action'], zero_division=0),
    'roc_auc': roc_auc_score(y_test_arr, t['baseline_score']),
    'p_at_10': precision_at_k(y_test_arr, t['baseline_score'], 10),
    'p_at_50': precision_at_k(y_test_arr, t['baseline_score'], 50),
    'p_at_200': precision_at_k(y_test_arr, t['baseline_score'], 200),
})

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = pipe.predict(X_test)
    rows.append({
        'method': name,
        'precision': precision_score(y_test_arr, pred, zero_division=0),
        'recall': recall_score(y_test_arr, pred, zero_division=0),
        'f1': f1_score(y_test_arr, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test_arr, proba),
        'p_at_10': precision_at_k(y_test_arr, proba, 10),
        'p_at_50': precision_at_k(y_test_arr, proba, 50),
        'p_at_200': precision_at_k(y_test_arr, proba, 200),
    })

comparison = pd.DataFrame(rows).set_index('method').round(3)
print(comparison)

Baseline action counts on test set:
baseline_action
1    6536
0    5233
Name: count, dtype: int64
                                               precision  recall     f1  \
method                                                                    
Baseline rule (train-calibrated, test-scored)      0.074   0.169  0.103   
logistic_regression                                0.601   0.869  0.710   
decision_tree_shallow                              0.588   1.000  0.741   
random_forest                                      0.588   1.000  0.740   

                                               roc_auc  p_at_10  p_at_50  \
method                                                                     
Baseline rule (train-calibrated, test-scored)    0.294      0.3     0.48   
logistic_regression                              0.900      1.0     0.84   
decision_tree_shallow                            0.905      0.4     0.62   
random_forest                                    0.908      0.9     0.9

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**Permutation importance (ROC-AUC drop):** CTR is the strongest feature in the Random Forest, with a mean ROC-AUC drop of 0.1514 when permuted. `log_clicks_15d` is the second strongest feature at 0.0876, while `avg_position_15d`, `log_impressions_15d`, and `position_tier` contribute much less individually. This suggests that the model's ranking signal is driven primarily by observed click-through behavior and click volume rather than the coarse position buckets used by the hand-built rule.

**Where the model is right and the rule is wrong:** 7,855 test rows fall into this group. These rows have median impressions of 71 and median CTR of 0.0, indicating that many of the model's gains occur on lower-volume, zero-click or near-zero-click content. The model appears to capture continuous differences in the available signals that the threshold-based rule does not.

**Where the rule is right and the model is wrong:** 1,429 test rows fall into this group. These rows are more visible, with median impressions of 485 and median CTR of 0.0073. This indicates that the model's remaining errors are more concentrated among comparatively successful pages, where distinguishing genuine decline from normal variation may be harder.

**Practical takeaway:** The model materially improves on the hand-built rule, but its errors are not random. Human review should be especially careful for higher-impression, higher-CTR pages because these are the cases where the baseline retained more correct calls than the model. The permutation-importance result also suggests that future iterations should investigate richer click-behavior signals rather than relying heavily on coarse position buckets.

In [6]:
from sklearn.inspection import permutation_importance

# Use whichever model Section 3's table actually justified — Random Forest had the best AUC
CHOSEN_MODEL = 'random_forest'
chosen = models[CHOSEN_MODEL]

perm = permutation_importance(chosen, X_test, y_test, n_repeats=20, random_state=42, scoring='roc_auc', n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False)
print('Permutation importance (ROC-AUC drop):')
print(importance.round(4))

t['model_pred'] = chosen.predict(X_test)
t['true_label'] = y_test_arr

model_right_rule_wrong = t[(t['model_pred'] == t['true_label']) & (t['baseline_action'] != t['true_label'])]
rule_right_model_wrong = t[(t['baseline_action'] == t['true_label']) & (t['model_pred'] != t['true_label'])]

print()
print(f'Model correct, rule wrong: {len(model_right_rule_wrong)} rows')
print(model_right_rule_wrong[['impressions_15d','avg_position_15d','ctr_15d','position_tier']].describe(include='all'))
print()
print(f'Rule correct, model wrong: {len(rule_right_model_wrong)} rows')
print(rule_right_model_wrong[['impressions_15d','avg_position_15d','ctr_15d','position_tier']].describe(include='all'))

Permutation importance (ROC-AUC drop):
ctr_15d                0.1514
log_clicks_15d         0.0876
avg_position_15d       0.0044
log_impressions_15d    0.0031
position_tier          0.0015
dtype: float64

Model correct, rule wrong: 7855 rows
        impressions_15d  avg_position_15d      ctr_15d position_tier
count       7855.000000       7855.000000  7855.000000          7855
unique              NaN               NaN          NaN             5
top                 NaN               NaN          NaN        page_1
freq                NaN               NaN          NaN          5136
mean         369.308466          8.271729     0.004882           NaN
std         1045.028648          5.925702     0.025283           NaN
min            1.000000          0.000000     0.000000           NaN
25%           12.000000          4.752798     0.000000           NaN
50%           71.000000          7.030303     0.000000           NaN
75%          296.000000          9.885904     0.004226           NaN

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.